# Stage 4 — Multi-label MLP Species Distribution Model

Trains the final SDM neural network on `sampled.parquet` and saves four artefacts:

| File | Contents |
|------|----------|
| `models/sdm_model.pt` | Weights + architecture metadata |
| `models/scaler.pkl` | Fitted `SDMScaler` (train statistics only) |
| `models/species_labels.json` | Ordered list of species names (col index → name) |
| `models/species_metadata.json` | `{taxon: {common, iconic_taxon}}` for the API |

**Architecture:** `[6 → 128 → 64 → 32 → S]` with BatchNorm + ReLU + Dropout(0.3)  
**Loss:** `BCEWithLogitsLoss` with per-species `pos_weight` (handles 50–500× class imbalance)  
**Optimiser:** Adam, lr=1e-3, weight_decay=1e-4  
**Early stopping:** patience=10, min_delta=1e-4 on val loss

**Correctness notes:**
- The scaler is fitted on `X_train` only — never on val, test, or the full dataset
- `pos_weight` is computed from `Y_train` only
- Stage 5 (`04_evaluation.ipynb`) provides the geographically unbiased AUC via 5×5 spatial CV
- The random split here is only for early stopping signal

In [ ]:
import sys
from pathlib import Path
_here = Path.cwd()
_root = _here.parent if _here.name == 'notebooks' else _here
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)-8s %(name)s  %(message)s',
    force=True,
)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
from sklearn.model_selection import train_test_split

from pipeline.features import (
    build_features, build_label_matrix_fast, build_scaler,
    get_species_list, build_species_metadata,
)
from pipeline.model import (
    SDMModel, compute_species_auc,
    save_species_labels, load_species_labels,
)
from config import (
    SAMPLED_PATH, MODEL_PATH, SCALER_PATH, SPECIES_LABELS_PATH, MODELS_DIR,
    INPUT_DIM, HIDDEN_DIMS, DROPOUT, LEARNING_RATE, BATCH_SIZE,
    MAX_EPOCHS, PATIENCE, MIN_DELTA, MIN_SPECIES_AUC,
)

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 110})

print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Setup complete')

## 1. Load Stage 2 output and build feature / label matrices

In [ ]:
df = pd.read_parquet(SAMPLED_PATH)
print(f'sampled.parquet: {len(df):,} rows  |  {df.columns.tolist()}')
print(f'Presences : {(df.presence==1).sum():,}')
print(f'Backgrounds: {(df.presence==0).sum():,}')
print()

# Feature matrix (N, 6)
X = build_features(df)
print(f'X.shape = {X.shape}  dtype={X.dtype}')

# Species list and label matrix (N, S)
species_list = get_species_list(df)
S = len(species_list)
print(f'Species: {S}')

Y = build_label_matrix_fast(df, species_list)
print(f'Y.shape = {Y.shape}  positives={int(Y.sum()):,}')

# Sanity: Y positives should equal presence count
assert int(Y.sum()) == int((df['presence']==1).sum()), 'Label matrix mismatch'
print('\n✓ Feature and label matrices verified')

## 2. Train / val / test split

Stratified random split (70 / 15 / 15) on the presence indicator.  
Stage 5 `04_evaluation.ipynb` provides the geographically unbiased AUC via 5×5 spatial cross-validation — the split here is only used for early stopping and a quick sanity AUC.

In [ ]:
N = len(X)
idx_all = np.arange(N)

# 70% train, 30% temp  (stratified on presence/background)
idx_train, idx_temp = train_test_split(
    idx_all, test_size=0.30, random_state=42,
    stratify=df['presence'].values,
)
# split temp 50/50 → 15% val, 15% test
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, random_state=42,
    stratify=df['presence'].values[idx_temp],
)

X_train, Y_train = X[idx_train], Y[idx_train]
X_val,   Y_val   = X[idx_val],   Y[idx_val]
X_test,  Y_test  = X[idx_test],  Y[idx_test]

print(f'Train : {len(X_train):>6,} rows  ({(Y_train.sum(axis=1)>0).mean()*100:.1f}% rows with ≥1 presence)')
print(f'Val   : {len(X_val):>6,} rows')
print(f'Test  : {len(X_test):>6,} rows')

# Verify presence counts maintained
for name, Y_split in [('train', Y_train), ('val', Y_val), ('test', Y_test)]:
    pct = (Y_split.sum(axis=1) > 0).mean() * 100
    print(f'  {name}: {Y_split.sum():.0f} positives  ({pct:.1f}% presence rows)')

## 3. Fit scaler on training data ONLY

In [ ]:
scaler = build_scaler()       # unfitted
X_train_s = scaler.fit(X_train).transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

print('Scaler statistics (computed from training data only):')
from pipeline.features import FEATURE_NAMES
for i, name in enumerate(FEATURE_NAMES):
    print(f'  {name:<15s}  mean={scaler.mean_[i]:>8.3f}  std={scaler.scale_[i]:>7.3f}')

# Training set should be ~0 mean, ~1 std after scaling
assert np.allclose(X_train_s.mean(axis=0), 0.0, atol=1e-5), 'Train mean not ~0'
assert np.allclose(X_train_s.std(axis=0),  1.0, atol=1e-2), 'Train std not ~1'
print('\n✓ Scaler correctness verified (train mean≈0, std≈1)')

## 4. Class imbalance — per-species presence counts

In [ ]:
pos_per_species = Y_train.sum(axis=0)  # (S,)
neg_per_species = len(Y_train) - pos_per_species
ratio = neg_per_species / np.maximum(pos_per_species, 1)

print(f'Median neg:pos ratio  : {np.median(ratio):.0f}x')
print(f'Max    neg:pos ratio  : {ratio.max():.0f}x  ({species_list[int(ratio.argmax())]})')
print(f'Min    neg:pos ratio  : {ratio.min():.0f}x  ({species_list[int(ratio.argmin())]})')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(pos_per_species, bins=30, color='steelblue', edgecolor='none')
axes[0].axvline(np.median(pos_per_species), color='red', ls='--',
                label=f'Median={np.median(pos_per_species):.0f}')
axes[0].set_title('Training positives per species')
axes[0].set_xlabel('Positive count'); axes[0].set_ylabel('Species')
axes[0].legend()

axes[1].hist(np.clip(ratio, 0, 300), bins=30, color='darkorange', edgecolor='none')
axes[1].set_title('Neg:Pos ratio per species (capped at 300)')
axes[1].set_xlabel('Ratio'); axes[1].set_ylabel('Species')
axes[1].axvline(200, color='red', ls='--', label='pos_weight cap (200)')
axes[1].legend()

plt.suptitle('Class imbalance — handled by BCEWithLogitsLoss pos_weight')
plt.tight_layout(); plt.show()

## 5. Train the model

Full training run with early stopping (patience=10).  
Typical wall time: ~2–5 min on GPU, ~15–20 min on CPU.

In [ ]:
model = SDMModel(
    n_species=S,
    species_list=species_list,
    input_dim=INPUT_DIM,
    hidden_dims=list(HIDDEN_DIMS),
    dropout=DROPOUT,
    lr=LEARNING_RATE,
    batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    min_delta=MIN_DELTA,
)
print(model)
print(f'Trainable parameters: {model.parameter_count():,}')

In [ ]:
history = model.fit(X_train_s, Y_train, X_val_s, Y_val)
print(f'\nTraining finished: {len(history["train_loss"])} epochs')
print(f'Best val loss: {min(history["val_loss"]):.4f}  (epoch {history["val_loss"].index(min(history["val_loss"]))+1})')

## 6. Training curves

In [ ]:
epochs = np.arange(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Loss curves
axes[0].plot(epochs, history['train_loss'], label='Train loss', color='steelblue', lw=1.5)
axes[0].plot(epochs, history['val_loss'],   label='Val loss',   color='tomato',    lw=1.5)
best_ep = int(np.argmin(history['val_loss'])) + 1
axes[0].axvline(best_ep, color='grey', ls='--', alpha=0.7, label=f'Best epoch ({best_ep})')
axes[0].set_title('Loss curves')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCEWithLogitsLoss')
axes[0].legend()

# Val loss zoom (last 40% of training)
cutoff = max(1, int(0.6 * len(epochs)))
axes[1].plot(epochs[cutoff:], history['train_loss'][cutoff:], label='Train', color='steelblue', lw=1.5)
axes[1].plot(epochs[cutoff:], history['val_loss'][cutoff:],   label='Val',   color='tomato',    lw=1.5)
axes[1].axvline(best_ep, color='grey', ls='--', alpha=0.7)
axes[1].set_title('Loss curves (last 40% of training)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('BCEWithLogitsLoss')
axes[1].legend()

plt.suptitle('SDM Model Training Curves', fontsize=12)
plt.tight_layout()
plt.savefig(_root / 'models' / 'training_curves.png', bbox_inches='tight', dpi=120)
plt.show()

# Check for overfitting
train_final = history['train_loss'][-1]
val_final   = history['val_loss'][-1]
gap = val_final - train_final
print(f'Final train loss : {train_final:.4f}')
print(f'Final val loss   : {val_final:.4f}')
print(f'Train/val gap    : {gap:.4f}  {"⚠ possible overfit" if gap > 0.1 else "✓ OK"}')

## 7. Per-species AUC on test set

In [ ]:
Y_prob_test = model.predict_proba(X_test_s)
auc_scores = compute_species_auc(Y_test, Y_prob_test, species_list, min_positives=5)

auc_valid = {sp: v for sp, v in auc_scores.items() if not np.isnan(v)}
auc_nan   = {sp: v for sp, v in auc_scores.items() if np.isnan(v)}

auc_arr = np.array(list(auc_valid.values()))
print(f'Species with AUC computed  : {len(auc_valid)}/{S}')
print(f'  Mean AUC  : {auc_arr.mean():.3f}')
print(f'  Median AUC: {np.median(auc_arr):.3f}')
print(f'  Min AUC   : {auc_arr.min():.3f}  ({min(auc_valid, key=auc_valid.get)})')
print(f'  Max AUC   : {auc_arr.max():.3f}  ({max(auc_valid, key=auc_valid.get)})')
print(f'Species below AUC threshold ({MIN_SPECIES_AUC}): {(auc_arr < MIN_SPECIES_AUC).sum()}')
print(f'Species skipped (few positives in test split): {len(auc_nan)}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# AUC histogram
axes[0].hist(auc_arr, bins=20, color='steelblue', edgecolor='none', alpha=0.85)
axes[0].axvline(MIN_SPECIES_AUC, color='red', ls='--',
                label=f'MIN_SPECIES_AUC={MIN_SPECIES_AUC}')
axes[0].axvline(np.median(auc_arr), color='green', ls='--',
                label=f'Median={np.median(auc_arr):.3f}')
axes[0].set_title(f'Per-species AUC on test set (n={len(auc_valid)})')
axes[0].set_xlabel('ROC-AUC'); axes[0].set_ylabel('Species count')
axes[0].legend()

# Sorted species AUC
sorted_items = sorted(auc_valid.items(), key=lambda x: x[1])
sorted_names = [x[0].split()[-1][:20] for x in sorted_items]  # genus abbreviation
sorted_aucs  = [x[1] for x in sorted_items]

colours = ['tomato' if a < MIN_SPECIES_AUC else 'steelblue' for a in sorted_aucs]
axes[1].barh(range(len(sorted_names)), sorted_aucs, color=colours, height=0.7)
axes[1].axvline(MIN_SPECIES_AUC, color='red', ls='--', lw=1)
axes[1].set_yticks(range(len(sorted_names)))
axes[1].set_yticklabels(sorted_names, fontsize=5)
axes[1].set_xlabel('ROC-AUC')
axes[1].set_title('Per-species AUC (red = below threshold)')

plt.suptitle('Test-set AUC — Stage 5 spatial CV gives geographically unbiased scores', fontsize=10)
plt.tight_layout()
plt.savefig(_root / 'models' / 'species_auc.png', bbox_inches='tight', dpi=120)
plt.show()

## 8. Top and bottom performing species

In [ ]:
sorted_auc_df = pd.DataFrame(
    [(sp, v, int(Y_train[:, species_list.index(sp)].sum())) 
     for sp, v in sorted(auc_valid.items(), key=lambda x: -x[1])],
    columns=['species', 'auc', 'train_positives']
)

print('Top 15 species (highest AUC):')
display(sorted_auc_df.head(15).to_string(index=False))
print()
print('Bottom 15 species (lowest AUC):')
display(sorted_auc_df.tail(15).to_string(index=False))
print()
print('Species below threshold (will be flagged by Stage 5 spatial CV):')
below = sorted_auc_df[sorted_auc_df['auc'] < MIN_SPECIES_AUC]
if len(below):
    display(below.to_string(index=False))
else:
    print('  None')

## 9. Prediction calibration check — 3 representative species

In [ ]:
from sklearn.calibration import calibration_curve

# Pick high / median / low AUC species (from those with enough test positives)
ranked = sorted_auc_df[sorted_auc_df['train_positives'] >= 10].reset_index(drop=True)
rep_indices = [
    0,                          # highest AUC
    len(ranked) // 2,           # median AUC
    len(ranked) - 1,            # lowest AUC
]
rep_species = [ranked.iloc[i]['species'] for i in rep_indices]
rep_aucs    = [ranked.iloc[i]['auc']     for i in rep_indices]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, sp, auc in zip(axes, rep_species, rep_aucs):
    j = species_list.index(sp)
    y_true = Y_test[:, j]
    y_prob = Y_prob_test[:, j]
    
    if y_true.sum() >= 5:
        frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy='quantile')
        ax.plot(mean_pred, frac_pos, 's-', color='steelblue', label='Model')
        ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect')
        ax.set_xlabel('Mean predicted probability')
        ax.set_ylabel('Fraction of positives')
    else:
        ax.text(0.5, 0.5, 'Insufficient\ntest positives', ha='center', va='center')
    
    ax.set_title(f'{sp.split()[0][0]}. {" ".join(sp.split()[1:])[:25]}\nAUC={auc:.3f}', fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle('Calibration curves (high / median / low AUC species)', y=1.02)
plt.tight_layout()
plt.show()

## 10. Save all artefacts

In [ ]:
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# 1) Model weights + architecture
model.save(MODEL_PATH)

# 2) Fitted scaler (train statistics only)
scaler.save(SCALER_PATH)

# 3) Species labels — ordered list (col index → species name)
save_species_labels(species_list, SPECIES_LABELS_PATH)

# 4) Species metadata — {taxon: {common_name, iconic_taxon}} for the API /species endpoint
SPECIES_METADATA_PATH = MODELS_DIR / 'species_metadata.json'
meta = build_species_metadata(df, species_list)
with open(SPECIES_METADATA_PATH, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'Saved species_metadata.json → {SPECIES_METADATA_PATH}')

# 5) AUC scores (useful for Stage 5 comparison and API filtering)
AUC_PATH = MODELS_DIR / 'species_auc_random_split.json'
with open(AUC_PATH, 'w') as f:
    json.dump({sp: (None if np.isnan(v) else round(v, 4)) for sp, v in auc_scores.items()}, f, indent=2)
print(f'Saved species AUC scores → {AUC_PATH}')

print('\nAll artefacts saved:')
for p in sorted(MODELS_DIR.iterdir()):
    print(f'  {p.name:<35s}  {p.stat().st_size/1024:.1f} KB')

## 11. ✅ Verify saved artefacts (round-trip load)

In [ ]:
from pipeline.features import SDMScaler

# Load model
model2 = SDMModel.load(MODEL_PATH)
assert model2.n_species == S, 'n_species mismatch after reload'
assert model2.species_list == species_list, 'species_list mismatch after reload'

# Load scaler
scaler2 = SDMScaler.load(SCALER_PATH)
assert np.allclose(scaler2.mean_, scaler.mean_), 'Scaler mean mismatch after reload'

# Predictions from reloaded model should be identical
probs2 = model2.predict_proba(scaler2.transform(X_test))
assert np.allclose(probs2, Y_prob_test, atol=1e-5), 'Reload predictions differ'

# Load species labels
labels2 = load_species_labels(SPECIES_LABELS_PATH)
assert labels2 == species_list, 'species_labels.json mismatch'

# Load metadata
with open(SPECIES_METADATA_PATH) as f:
    meta2 = json.load(f)
assert set(meta2.keys()) == set(species_list), 'Metadata species set mismatch'

print('✓ All artefacts round-trip verified')
print(f'\nModel: {model2}')
print(f'\nSample metadata entries:')
for sp in species_list[:3]:
    print(f'  {sp}: {meta2[sp]}')

## 12. Summary

Run this cell for a compact stage report before proceeding to Stage 5.

In [ ]:
auc_arr_all = np.array([v for v in auc_scores.values() if not np.isnan(v)])
n_above = (auc_arr_all >= MIN_SPECIES_AUC).sum()
n_below = (auc_arr_all < MIN_SPECIES_AUC).sum()

sep = '=' * 64
lines = [
    '',
    sep,
    '  STAGE 4 — MODEL TRAINING SUMMARY',
    sep,
    f'  Architecture    : {INPUT_DIM} → {" → ".join(str(h) for h in HIDDEN_DIMS)} → {S}',
    f'  Parameters      : {model.parameter_count():,}',
    f'  Training epochs : {len(history["train_loss"])} (early stop)',
    f'  Best val loss   : {min(history["val_loss"]):.4f}',
    f'  Device          : {model.device}',
    '',
    f'  Species total   : {S}',
    f'  AUC computed    : {len(auc_arr_all)} (≥5 test positives)',
    f'  Mean test AUC   : {auc_arr_all.mean():.3f}',
    f'  Median test AUC : {np.median(auc_arr_all):.3f}',
    f'  ≥ {MIN_SPECIES_AUC} threshold    : {n_above} species',
    f'  < {MIN_SPECIES_AUC} threshold    : {n_below} species  (Stage 5 spatial CV may exclude)',
    '',
    '  Artefacts saved:',
    f'    {MODEL_PATH.name}',
    f'    {SCALER_PATH.name}',
    f'    {SPECIES_LABELS_PATH.name}',
    '    species_metadata.json',
    sep,
    '',
    '  → Proceed to 04_evaluation.ipynb (Stage 5 spatial CV)',
    sep,
]
print('\n'.join(lines))